In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

ה-primitive של Executor הוא חלק מ[מודל ההרצה המכוונת](/guides/directed-execution-model), שמספק גמישות רבה יותר בהתאמה אישית של תהליך עבודת הפחתת שגיאות.

הקלטים והפלטים של ה-primitive של Executor שונים מאוד מאלו של ה-primitives של [Sampler](/guides/sampler-input-output) ו-[Estimator](/guides/estimator-input-output). לדוגמה, במקום לקבל רשימת PUBs כקלט, Executor מקבל `QuantumProgram`, שמכיל רשימת אובייקטי `QuantumProgramItem`. מחלקות מכילות אלה נותנות לך גמישות רבה יותר מ-PUB, שהוא מבנה נתונים tuple פשוט.

הפלט של Executor הוא `QuantumProgramResult`, שהוא iterable ומכיל אלמנט אחד לכל `QuantumProgramItem` קלט.

<span id="programs"></span>

## קלטים: תוכניות קוונטיות

כפי שצוין קודם, הקלט ל-primitive של Executor הוא [`QuantumProgram`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgram.html#QuantumProgram), שהוא iterable של אובייקטי [`QuantumProgramItem`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramItem.html). אובייקטים אלה יכולים להיות משני סוגים:
- `CircuitItem`, שבדרך כלל מאחסן מעגל וערכי הפרמטרים שלו (אם יש).
- `SamplexItem`, שבדרך כלל מאחסן את הבאים:
    - מעגל template
    - אובייקט samplex, שמשמש לייצור סדרות פרמטרים אקראיות בזמן ריצה (לדוגמה לביצוע twirling או הזרקת רעש)
    - ארגומנטים עבור ה-samplex, שעשויים לכלול ערכי פרמטרים עבור המעגל המקורי

כל אחד מהפריטים הללו מייצג משימה שונה שעל Executor לבצע.

### לפני שמתחילים

חלק מדוגמאות הקוד בדף זה משתמשות ב-`samplex`, שהוא חלק מחבילת Samplomatic. לכן, לפני הרצת בלוקי קוד אלה, חייבים להתקין את Samplomatic, כפי שמוצג בבלוק הקוד הבא. למידע נוסף, ראו את [תיעוד Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit_ibm_runtime import Executor, QiskitRuntimeService
from qiskit.circuit import Parameter, QuantumCircuit
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Initialize an empty program
program = QuantumProgram(shots=1024)

# Initialize and transpile a 3-qubit quantum circuit with 2 parameters.
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)

# `measure_all` adds a 3-bit classical register named "meas"
circuit.measure_all()

# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Generate a preset pass manager
# This will be used to convert the abstract circuit to an
# equivalent Instruction Set Architecture (ISA) circuit.
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Transpile the circuit
isa_circuit = preset_pass_manager.run(circuit)

### דוגמה: יצירת `QuantumProgram` עם שתי משימות שונות
ראשית אתחל את התוכנית הקוונטית שלך, ואז הוסף פריטי תוכנית אליה באמצעות `append_circuit_item` או `append_samplex_item` (אם יש samplex), כפי שמוצג בדוגמאות הבאות.

התא הבא מאתחל `QuantumProgram` ומציין שצריך להריץ 1024 shots עבור כל תצורה של כל פריט בתוכנית.

> **Note:** בשונה מ-Sampler, `QuantumProgram` מקבל רק ערך shot בודד. אם רוצים ערך shot שונה, צריך `QuantumProgram` נפרד, שיהיה עבודה נפרדת.

In [2]:
# Append the transpiled circuit and an array
# containing 10 sets of parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(
        10, 2
    ),  # 10 sets of parameter values and 2 parameters
)

### הוספת `CircuitItem`
לאחר מכן, הוסף את מעגל היעד, שבוצע עליו transpile לפי ה-instruction set architecture (ISA) של ה-backend, ל-`QuantumProgram`. מכיוון שלמעגל זה יש שני פרמטרים, עלינו גם לספק את ערכי הפרמטרים (10 סדרות בדוגמה זו). הרצת `CircuitItem` זה היא המשימה הראשונה שהתוכנית תבצע.

In [3]:
# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Use the boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template circuit and the samplex.  The template circuit has parametric gates
# without fixed values and the samplex randomly generates the parameter
# values on the server side at runtime to perform twirling.
template_circuit, samplex = build(boxed_circuit)

# Determine what arguments are required by the samplex.
# Input the arguments in samplex_arguments.
print(samplex.inputs())

TensorInterface(<
  - 'parameter_values' <float64[2]>: Input parameter values to use during sampling.
>)


In [4]:
# Append the template circuit and samplex as a samplex item
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        # the arguments required by the samplex.sample method
        "parameter_values": np.random.rand(10, 2),
    },
    shape=(28, 10),  # 28 randomizations and 10 sets of parameter values
)

In [5]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

## Outputs

Executor's output is a [`QuantumProgramResult`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramResult.html#QuantumProgramResult), which is an iterable. It contains one entry per input `QuantumProgramItem` in the same order as the input items. Each of these output items is a dictionary where the keys are strings that correspond the classical registers' names in the input circuits (among others), so you no longer need to memorize these names like you did with Sampler output. The dictionary values are of type `np.ndarray`.

The result for the previous example contains these items:

### `CircuitItem` result

The first item contains the results of running the first task (a `CircuitItem`) in the program. It contains a single key, `meas`, which is the name of the classical register in the input circuit. The value of this key maps to an `np.ndarray` of shape `(parameter sets, shots, register bits)`, which is (10, 1024, 3) for the above example.

The following code illustrates how to access this information:

In [6]:
# Access the results of the classical register of task #0, a CircuitItem
result_0 = result[0]["meas"]
print(f"Result shape: {result_0.shape}")

Result shape: (10, 1024, 3)


### `SamplexItem` result

The second item contains the results of running the second task (a `SamplexItem`) in the program. This item contains multiple keys. The `meas` key, which is the name of the input circuit's classical register, maps to that register's array of results. This array has the shape `(randomizations, parameter sets, shots, classical bits)`, or (28, 10, 1024, 3) in this example. Additionally, the output contains a `measurement_flips.meas` key, which is the bit-flip corrections to undo the measurement twirling for the `meas` register.  This output shape will be (28, 10, 1, 3) for our example because only one shot is required to perform the bit-flip.

In [7]:
# Access the results of the classical register of task #1
result_1 = result[1]["meas"]
print(f"Result shape: {result_1.shape}")

# Access the bit-flip corrections
flips_1 = result[1]["measurement_flips.meas"]
print(f"Bit-flip corrections shape: {flips_1.shape}")

# Undo the bit flips via classical XOR
unflipped_result_1 = result_1 ^ flips_1

Result shape: (28, 10, 1024, 3)
Bit-flip corrections shape: (28, 10, 1, 3)


## פלטים
הפלט של Executor הוא [`QuantumProgramResult`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/QuantumProgramResult.html#QuantumProgramResult), שהוא iterable. הוא מכיל ערך אחד לכל `QuantumProgramItem` קלט באותו סדר כמו פריטי הקלט. כל אחד מפריטי הפלט הללו הוא מילון שבו המפתחות הם מחרוזות המתאימות לשמות הרגיסטרים הקלאסיים במעגלי הקלט (בין היתר), כך שכבר אין צורך לזכור שמות אלה כפי שהיה צריך עם פלט Sampler. ערכי המילון הם מסוג `np.ndarray`.

התוצאה עבור הדוגמה הקודמת מכילה את הפריטים הבאים:

### תוצאת `CircuitItem`
הפריט הראשון מכיל את תוצאות הרצת המשימה הראשונה (`CircuitItem`) בתוכנית. הוא מכיל מפתח בודד, `meas`, שהוא שם הרגיסטר הקלאסי במעגל הקלט. ערך מפתח זה ממפה ל-`np.ndarray` בצורה `(parameter sets, shots, register bits)`, שהיא (10, 1024, 3) בדוגמה לעיל.

הקוד הבא ממחיש כיצד לגשת למידע זה: